# Step 5: Analysis Actor

Kilian Lüders & Hannah Birkenkötter

Analyzing the data annotated by the LLM.

**Steps:**
1. Check LLM Data
2. Merge Hand-Coding
3. Data Analysis

**Input:**
- `data/subset_demand_request_decide_llm.pkl` - (from `4_llm_inference.ipynb`)
- `data/actors_handcoded.xlsx` - handcoding actor
- `data/actors_handcoding_check.xlsx` - handcoded paragrpahs with missing actor coding

In [1]:
import numpy as np
import pandas as pd

### 1. Check LLM Data

In [2]:
data_llm = pd.read_pickle("data/subset_demand_request_decide_llm.pkl")
print(data_llm.shape)
data_llm = data_llm[data_llm.doc != "S_RES_0713_1991"] # delete false positive
print(data_llm.shape)

(1236, 23)
(1235, 23)


In [3]:
data_llm = data_llm[["text", "doc", "actors"]].copy()
data_llm = data_llm.rename(columns={'actors': 'actor_llm'})
data_llm['actor_llm_count'] = data_llm.actor_llm.apply(lambda x: len(x))

In [4]:
data_llm

,text,doc,actor_llm,actor_llm_count
1,Decides to keep the situation in Spain under c...,S_RES_0007_1946,"[Security Council, Any member of the Security ...",2
3,Requests that Government to submit to the Secu...,S_RES_0059_1948,[Government],1
13,Requests the Secretary-General to make availab...,S_RES_0067_1949,[Secretary-General],1
7,Requests the United Nations Commission on Kore...,S_RES_0082_1950,[United Nations Commission on Korea],1
6,Requests the United States to designate the co...,S_RES_0084_1950,[United States],1
...,...,...,...,...
34,Requests the following reports from the Secret...,S_RES_2705_2023,[Secretary-General],1
20,"Agrees to consider on a case-by-case basis, wh...",S_RES_2719_2023,"[African Union Peace and Security Council, Uni...",2
37,"Requests the Secretary-General, in consultatio...",S_RES_2719_2023,"[Secretary-General, Chairperson of the African...",2
38,Also requests the Secretary-General to submit ...,S_RES_2719_2023,[Secretary-General],1


### 2. Merge Hand-Coding

In [5]:
# handcoding actor
handcoding = pd.read_excel("data/actors_handcoded.xlsx").drop(columns=["Unnamed: 0"])
handcoding_dict = handcoding.set_index("actor").to_dict()['coding']

In [6]:
# handcoding cases with missing actor coding
handcoding_unclear = pd.read_excel("data/actors_handcoding_check.xlsx").drop(columns=["Unnamed: 0"])
handcoding_unclear = handcoding_unclear[['text', 'doc', 'actor_coding']]
handcoding_unclear['actor_coding'] = handcoding_unclear.actor_coding.apply(lambda x: [x])

In [7]:
data_llm['actor_coding'] = data_llm.actor_llm.apply(lambda x: [handcoding_dict[e] for e in x])
data_llm['actor_coding'] = data_llm.actor_coding.apply(lambda x: list(set(x)))

In [8]:
actor_data = data_llm.merge(handcoding_unclear, how="left", left_on=['text', 'doc'], right_on=['text', 'doc'], suffixes=('', '_hand'))

actor_data["actor_coding"] = np.where(actor_data["actor_coding_hand"].notna(), actor_data["actor_coding_hand"], actor_data["actor_coding"])
actor_data = actor_data.drop(columns=['actor_coding_hand'])

actor_data["actor_coding"] = actor_data.actor_coding.apply(lambda x: list(set(x)))
actor_data['actor_coding_count'] = actor_data.actor_coding.apply(lambda x: len(x))


In [9]:
# final dataset with hand codings
actor_data

,text,doc,actor_llm,actor_llm_count,actor_coding,actor_coding_count
0,Decides to keep the situation in Spain under c...,S_RES_0007_1946,"[Security Council, Any member of the Security ...",2,"[States and state actors, UN organ or entity]",2
1,Requests that Government to submit to the Secu...,S_RES_0059_1948,[Government],1,[States and state actors],1
2,Requests the Secretary-General to make availab...,S_RES_0067_1949,[Secretary-General],1,[UN organ or entity],1
3,Requests the United Nations Commission on Kore...,S_RES_0082_1950,[United Nations Commission on Korea],1,[UN organ or entity],1
4,Requests the United States to designate the co...,S_RES_0084_1950,[United States],1,[States and state actors],1
...,...,...,...,...,...,...
1230,Requests the following reports from the Secret...,S_RES_2705_2023,[Secretary-General],1,[UN organ or entity],1
1231,"Agrees to consider on a case-by-case basis, wh...",S_RES_2719_2023,"[African Union Peace and Security Council, Uni...",2,"[UN organ or entity, non-UN IO (includes Speci...",2
1232,"Requests the Secretary-General, in consultatio...",S_RES_2719_2023,"[Secretary-General, Chairperson of the African...",2,"[UN organ or entity, non-UN IO (includes Speci...",2
1233,Also requests the Secretary-General to submit ...,S_RES_2719_2023,[Secretary-General],1,[UN organ or entity],1


### 3. Data Analysis

In [10]:
# number of resolutions
actor_data.doc.nunique()

347

In [11]:
# number of actors
actor_data.actor_llm.explode().nunique()

565

In [12]:
def check_sec_general(actor_list):
    if type(actor_list) != list:
        return np.nan
    if len(actor_list) == 1:
        if "Secretary-General" in actor_list[0]:
            return True
    return False


actor_data['only_secretary_general'] = actor_data.actor_llm.apply(lambda X: check_sec_general(X))

actor_data.only_secretary_general.value_counts()

only_secretary_general
False    818
True     417
Name: count, dtype: int64

In [13]:
def check_un_system(actor_list):
    if type(actor_list) != list:
        return np.nan
    if len(actor_list) == 1:
        if "UN organ or entity" in actor_list[0]:
            return True
    return False


actor_data['only_UN_system'] = actor_data.actor_coding.apply(lambda X: check_un_system(X))

actor_data.only_UN_system.value_counts()

only_UN_system
True     757
False    478
Name: count, dtype: int64

In [14]:
def check_states(actor_list):
    if type(actor_list) != list:
        return np.nan
    if len(actor_list) == 1:
        if "States" in actor_list[0]:
            return True
    return False


actor_data['only_states'] = actor_data.actor_coding.apply(lambda X: check_states(X))

actor_data.only_states.value_counts()

only_states
False    1123
True      112
Name: count, dtype: int64

In [15]:
non_un_actor_data = actor_data[~(actor_data.only_UN_system)]
non_un_actor_data.actor_coding.explode().value_counts()

actor_coding
States and state actors                                    310
UN organ or entity                                         232
Unspecified/mixed (e.g. "parties")                         146
non-UN IO (includes Specialized Agencies)                   50
Armed Non-State Actor                                       27
States and State Actors, including specific governments     16
Civil Society Actor                                         11
De-facto regime                                              8
FEHLER not an addressee                                      7
Non-UN IO (includes Specialized Agencies)                    2
Unspecified/mixed ("parties" "international community")      1
Name: count, dtype: int64